# Northwind Traders — Executive Report (Databricks)

This notebook consolidates the 5 analytical views from the dashboard to support executive decision-making.

Scope:
- Page 1: Executive Overview
- Page 2: Avg Order Value & Product Mix
- Page 3: Customers and Churn
- Page 4: Commercial Performance
- Page 5: Logistics

Data source: workspace.indicium_gold on Databricks SQL Warehouse.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from datetime import datetime, timedelta
import warnings
import os
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from databricks import sql
import plotly.io as pio
pio.templates.default = 'plotly_white'

# Load .env
env_file = r"c:\Users\rafae\Documents\Carreira\challenge_indicium\.env"
load_dotenv(env_file)

print("✅ All libraries loaded")

✅ Todas as bibliotecas carregadas


In [ ]:
# Conectar ao Databricks
DATABRICKS_HOST = os.getenv("DATABRICKS_HOST")
DATABRICKS_HTTP_PATH = os.getenv("DATABRICKS_HTTP_PATH")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")

conn = sql.connect(
    server_hostname=DATABRICKS_HOST,
    http_path=DATABRICKS_HTTP_PATH,
    personal_access_token=DATABRICKS_TOKEN
)
print("✅ Conectado ao Databricks")

✅ Conectado ao Databricks


In [ ]:
# Load tables from Databricks
def load_table(table_name):
    query = f"SELECT * FROM workspace.indicium_gold.{table_name}"
    return pd.read_sql(query, con=conn)

print("⏳ Loading data...")
fact_sales = load_table("fact_sales")
dim_calendar = load_table("dim_calendar")
dim_customer = load_table("dim_customer")
dim_product = load_table("dim_product")   # includes category_name and category_description
dim_employee = load_table("dim_employee")
dim_territory = load_table("dim_territory")  # includes region_description
bridge_employee_territory = load_table("bridge_employee_territory")

# Parse main date columns
for col in ["order_date", "required_date", "shipped_date"]:
    if col in fact_sales.columns:
        fact_sales[col] = pd.to_datetime(fact_sales[col], errors="coerce")

print("✅ Data loaded successfully!")
print(f"   • fact_sales:              {len(fact_sales):,} records")
print(f"   • dim_product (w/ categ.): {len(dim_product):,} records")
print(f"   • dim_customer:            {len(dim_customer):,} records")
print(f"   • dim_employee:            {len(dim_employee):,} records")
print(f"   • dim_territory:           {len(dim_territory):,} records")
print(f"   • dim_calendar:            {len(dim_calendar):,} records")

⏳ Carregando dados...
✅ Dados carregados com sucesso!
   • fact_sales:              2,155 registros
   • dim_product (c/ categ.): 77 registros
   • dim_customer:            91 registros
   • dim_employee:            9 registros
   • dim_territory:           53 registros
   • dim_calendar:            672 registros


---
# PAGE 1: EXECUTIVE OVERVIEW
Main KPIs, monthly revenue and revenue by category

In [ ]:
# Executive Overview KPIs
receita_liquida = fact_sales['net_sales'].sum()
receita_bruta = fact_sales['gross_sales'].sum()
total_pedidos = fact_sales['order_id'].nunique()
ticket_medio = receita_liquida / total_pedidos
total_clientes = fact_sales['customer_id'].nunique()

print("\n" + "="*60)
print("📊 EXECUTIVE OVERVIEW - KPIs")
print("="*60)
print(f"Net Revenue:        R$ {receita_liquida:>15,.2f}")
print(f"Gross Revenue:      R$ {receita_bruta:>15,.2f}")
print(f"Total Orders:          {total_pedidos:>15,}")
print(f"Avg Order Value:    R$ {ticket_medio:>15,.2f}")
print(f"Active Customers:      {total_clientes:>15,}")
print("="*60)


📊 VISÃO EXECUTIVA - KPIs
Receita Líquida:    R$    1,265,793.04
Receita Bruta:      R$    1,354,458.59
Total de Pedidos:                  830
Ticket Médio:       R$        1,525.05
Clientes Ativos:                    89


In [ ]:
# Fig 1: Monthly Revenue
receita_mes = fact_sales.groupby(fact_sales['order_date'].dt.to_period('M')).agg({
    'net_sales': 'sum',
    'order_id': 'nunique'
}).reset_index()
receita_mes['data'] = receita_mes['order_date'].dt.to_timestamp()
receita_mes = receita_mes.drop('order_date', axis=1).rename(columns={'net_sales': 'receita', 'order_id': 'pedidos'})

fig = go.Figure()
fig.add_trace(go.Scatter(x=receita_mes['data'], y=receita_mes['receita'], mode='lines+markers', 
                         name='Revenue', line=dict(color='#1f77b4', width=3), marker=dict(size=8)))
fig.update_layout(title='📈 Monthly Net Revenue', xaxis_title='Period', yaxis_title='Revenue (R$)', height=400)
fig.show()

In [ ]:
# Fig 2: Revenue by Category  (category_name now inline in dim_product)
fact_cat = fact_sales.merge(dim_product[['product_id', 'category_name']], on='product_id', how='left')
receita_cat = fact_cat.groupby('category_name')['net_sales'].sum().sort_values(ascending=False)

def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'

fig = px.bar(x=receita_cat.values, y=receita_cat.index, orientation='h',
             title='📊 Revenue by Category', labels={'x': 'Revenue (R$)', 'y': 'Category'},
             color=receita_cat.values, color_continuous_scale='Viridis',
             text=[fmt_cur_short(v) for v in receita_cat.values])
fig.update_traces(textposition='inside', insidetextanchor='middle', cliponaxis=False)
fig.update_layout(height=400, showlegend=False, uniformtext_minsize=10, uniformtext_mode='hide')
fig.update_yaxes(autorange='reversed')
fig.show()

---
# PAGE 2: AVG ORDER VALUE & PRODUCT MIX
Discount analysis, product mix and top products

In [ ]:
# Page 2 KPIs
itens_vendidos = fact_sales['quantity'].sum()
desconto_total = fact_sales['discount_value'].sum()
receita_por_cliente = receita_liquida / total_clientes

print("\n" + "="*60)
print("🛍️  AVG ORDER VALUE & PRODUCT MIX")
print("="*60)
print(f"Items Sold:              {itens_vendidos:>15,.0f}")
print(f"Total Discount:      R$ {desconto_total:>15,.2f}")
print(f"Revenue per Customer:R$ {receita_por_cliente:>15,.2f}")
print("="*60)


🛍️  TICKET E MIX DE PRODUTOS
Itens Vendidos:                   51,317
Desconto Concedido:  R$       88,665.55
Receita por Cliente: R$       14,222.39


In [ ]:
# Fig 3: Discount vs Order Value (scatter)
pedidos_desc = fact_sales.groupby('order_id').agg({'net_sales': 'sum', 'discount': 'mean'}).reset_index()

fig = px.scatter(pedidos_desc, x='discount', y='net_sales',
                title='💰 Discount vs. Order Value', 
                labels={'discount': 'Discount Rate', 'net_sales': 'Value (R$)'},
                opacity=0.6, color='discount', color_continuous_scale='Reds')
fig.update_layout(height=400)
fig.show()

In [ ]:
# Fig 4: Top 15 Products
fact_prod = fact_sales.merge(dim_product[['product_id', 'product_name']], on='product_id', how='left')
top_prod = fact_prod.groupby('product_name')['net_sales'].sum().sort_values(ascending=False).head(15)

def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'

fig = px.bar(x=top_prod.values, y=top_prod.index, orientation='h',
             title='🏆 Top 15 Products by Revenue',
             labels={'x': 'Revenue (R$)', 'y': 'Product'},
             color=top_prod.values, color_continuous_scale='Blues',
             text=[fmt_cur_short(v) for v in top_prod.values])
fig.update_traces(textposition='inside', insidetextanchor='middle', cliponaxis=False)
fig.update_layout(height=500, showlegend=False, uniformtext_minsize=9, uniformtext_mode='hide')
fig.update_yaxes(autorange='reversed')
fig.show()

---
# PAGE 3: CUSTOMERS AND CHURN
Retention analysis, inactivity and segmentation

In [ ]:
# Churn and Inactivity Analysis
data_ref = fact_sales['order_date'].max()
primeiro_pedido = fact_sales.groupby('customer_id')['order_date'].min()
ultimo_pedido = fact_sales.groupby('customer_id')['order_date'].max()
dias_inativo = (data_ref - ultimo_pedido).dt.days
clientes_risco = (dias_inativo >= 90).sum()
taxa_retencao = ((total_clientes - clientes_risco) / total_clientes * 100)
novos_clientes = (primeiro_pedido == primeiro_pedido.max()).sum()

print("\n" + "="*60)
print("👥 CUSTOMERS AND CHURN")
print("="*60)
print(f"Active Customers:          {total_clientes:>15,}")
print(f"New Customers (latest):    {novos_clientes:>15,}")
print(f"At-Risk Customers (90+d):  {clientes_risco:>15,}")
print(f"Retention Rate:              {taxa_retencao:>14.2f}%")
print("="*60)


👥 CLIENTES E CHURN
Clientes Ativos:                        89
Clientes Novos (últimas):                1
Clientes em Risco (90+d):               16
Taxa de Retenção:                     82.02%


In [ ]:
# Fig 5: Inactivity Distribution
dias_df = pd.DataFrame({'dias_inativo': dias_inativo})
fig = px.histogram(dias_df, x='dias_inativo', nbins=40,
                  title='📉 Days Without a Purchase — Distribution',
                  labels={'dias_inativo': 'Days Inactive', 'count': 'Customers'},
                  color_discrete_sequence=['#636EFA'])
fig.add_vline(x=90, line_dash='dash', line_color='red', annotation_text='At-Risk (90d)')
fig.update_layout(height=400)
fig.show()

---
# PAGE 4: COMMERCIAL PERFORMANCE
Salesperson ranking and territorial analysis

In [ ]:
# Salesperson Analysis
fact_emp = fact_sales.merge(dim_employee[['employee_id', 'first_name', 'last_name']], on='employee_id', how='left')
fact_emp['salesperson'] = fact_emp['first_name'].fillna('') + ' ' + fact_emp['last_name'].fillna('')
receita_vendedor = fact_emp.groupby('salesperson')['net_sales'].sum().sort_values(ascending=False)
pedidos_vendedor = fact_emp.groupby('salesperson')['order_id'].nunique().sort_values(ascending=False)

print("\n" + "="*60)
print("💼 COMMERCIAL PERFORMANCE")
print("="*60)
print(f"Top Salesperson: {receita_vendedor.index[0]} (R$ {receita_vendedor.values[0]:,.2f})")
print("\nTop 5 by Revenue:")
for i, (v, r) in enumerate(receita_vendedor.head(5).items(), 1):
    print(f"  {i}. {v:40s} R$ {r:>12,.2f}")
print("="*60)


💼 COMERCIAL E TERRITÓRIO
Top Vendedor: Margaret Peacock (R$ 232,890.85)

Top 5 por Receita:
  1. Margaret Peacock                         R$   232,890.85
  2. Janet Leverling                          R$   202,812.84
  3. Nancy Davolio                            R$   192,107.60
  4. Andrew Fuller                            R$   166,537.76
  5. Laura Callahan                           R$   126,862.28


In [ ]:
# Fig 6: Salesperson Ranking
def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'

fig = px.bar(x=receita_vendedor.head(10).values, y=receita_vendedor.head(10).index, orientation='h',
             title='🏅 Top 10 Salespersons by Revenue',
             labels={'x': 'Revenue (R$)', 'y': 'Salesperson'},
             color=receita_vendedor.head(10).values, color_continuous_scale='Plasma',
             text=[fmt_cur_short(v) for v in receita_vendedor.head(10).values])
fig.update_traces(textposition='inside', insidetextanchor='middle', cliponaxis=False)
fig.update_layout(height=400, showlegend=False, uniformtext_minsize=9, uniformtext_mode='hide')
fig.update_yaxes(autorange='reversed')
fig.show()

---
# PAGE 5: LOGISTICS
Delivery lead time analysis and impact

In [ ]:
# Logistics KPIs
pedidos_prazo = (fact_sales['shipped_on_time_flag'] == 1).sum()
total_pedidos_entrega = len(fact_sales)
pct_prazo = (pedidos_prazo / total_pedidos_entrega * 100)
atraso_medio = fact_sales['shipping_delay_days'].mean()
receita_atraso = fact_sales[fact_sales['shipping_delay_days'] > 0]['net_sales'].sum()

print("\n" + "="*60)
print("📦 LOGISTICS")
print("="*60)
print(f"On-Time Orders:           {pedidos_prazo:>15,}")
print(f"% On-Time Deliveries:        {pct_prazo:>14.2f}%")
print(f"Avg Delay (days):            {atraso_medio:>14.2f}")
print(f"Revenue w/ Delay:       R$ {receita_atraso:>15,.2f}")
print("="*60)


📦 LOGÍSTICA
Pedidos no Prazo:                   1,990
% Entregas no Prazo:                  92.34%
Atraso Médio (dias):                 -19.47
Receita c/ Atraso:      R$       66,535.61


In [16]:
# Fig 7: Gauge % Prazo
fig = go.Figure(go.Indicator(
    mode='gauge+number+delta',
    value=pct_prazo,
    title={'text': '% Entregas on Top'},
    delta={'reference': 95},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': '#636EFA'},
        'steps': [
            {'range': [0, 50], 'color': '#f0f0f0'},
            {'range': [50, 90], 'color': '#f5f5f5'}
        ]
    }
))
fig.update_layout(height=400)
fig.show()

In [ ]:
# Fig 8: Delay Distribution
fig = px.histogram(fact_sales, x='shipping_delay_days', nbins=40,
                  title='📊 Delays and Early Deliveries — Distribution',
                  labels={'shipping_delay_days': 'Delay (days, negative = early)', 'count': 'Orders'},
                  color_discrete_sequence=['#AB63FA'])
fig.add_vline(x=0, line_dash='dash', line_color='green', annotation_text='On Time')
fig.update_layout(height=400)
fig.show()

---
# CONSOLIDATED SUMMARY

In [ ]:
print("\n" + "#"*65)
print("#" + " "*63 + "#")
print("#  NORTHWIND ANALYSIS - EXECUTIVE SUMMARY" + " "*22 + "#")
print("#" + " "*63 + "#")
print("#"*65)

print("\n📊 EXECUTIVE OVERVIEW")
print(f"   Net Revenue:    R$ {receita_liquida:>15,.2f}")
print(f"   Orders:            {total_pedidos:>15,}")
print(f"   Avg Order Value:R$ {ticket_medio:>14,.2f}")
print(f"   Customers:         {total_clientes:>15,}")

print("\n🛍️  AVG ORDER VALUE & MIX")
print(f"   Items Sold:        {itens_vendidos:>20,.0f}")
print(f"   Total Discount: R$ {desconto_total:>14,.2f}")
print(f"   Revenue/Customer:R$ {receita_por_cliente:>13,.2f}")

print("\n👥 CUSTOMERS AND CHURN")
print(f"   Retention Rate:    {taxa_retencao:>22.2f}%")
print(f"   At-Risk Customers: {clientes_risco:>15,}")
print(f"   New Customers:     {novos_clientes:>19,}")

print("\n💼 COMMERCIAL")
print(f"   Top Salesperson: {receita_vendedor.index[0]:>18s}")
print(f"   Active Salespersons: {len(receita_vendedor):>14,}")

print("\n📦 LOGISTICS")
print(f"   % On-Time Deliveries:  {pct_prazo:>17.2f}%")
print(f"   Avg Delay:             {atraso_medio:>24.2f} days")
print(f"   Revenue w/ Delay:   R$ {receita_atraso:>12,.2f}")

print("\n" + "#"*65)
print("\n✅ Analysis completed successfully from Databricks!")


#################################################################
#                                                               #
#  ANÁLISE NORTHWIND - RESUMO EXECUTIVO                        #
#                                                               #
#################################################################

📊 VISÃO EXECUTIVA
   Receita Líquida: R$    1,265,793.04
   Pedidos:                       830
   Ticket Médio: R$       1,525.05
   Clientes:                         89

🛍️  TICKET E MIX
   Itens Vendidos:               51,317
   Desconto Total: R$      88,665.55
   Receita/Cliente: R$     14,222.39

👥 CLIENTES E CHURN
   Taxa Retenção:                  82.02%
   Clientes em Risco:              16
   Novos Clientes:                   1

💼 COMERCIAL
   Top Vendedor:   Margaret Peacock
   Vendedores Ativos:              9

📦 LOGÍSTICA
   % Entregas no Prazo:             92.34%
   Atraso Médio:                   -19.47 dias
   Receita c/ Atraso: R$    66,535.61



---
# Export for Delivery

This section generates the notebook HTML and attempts to export it as PDF automatically.
If automatic PDF generation fails due to a missing local dependency, use the generated HTML and print > Save as PDF in the browser.

In [19]:
%pip install nbconvert -q
print('nbconvert instalado')

Note: you may need to restart the kernel to use updated packages.
nbconvert instalado


In [20]:
%pip install playwright -q
import subprocess, sys
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=False)
print('playwright/chromium instalados (ou ja existentes)')

Note: you may need to restart the kernel to use updated packages.
playwright/chromium instalados (ou ja existentes)


---
# Executive Presentation (Code-Free)

This section generates an executive-friendly presentation without any code blocks — only narrative, charts and results.
Expected outputs:
- PDF: analise_northwind_executivo.pdf

In [ ]:
import subprocess
import sys
from pathlib import Path
from html import escape

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

notebook_name = "analise_northwind_databricks.ipynb"
candidate_paths = [
    Path.cwd() / notebook_name,
    Path(r"c:\Users\rafae\Documents\Carreira\challenge_indicium") / notebook_name,
]

notebook_file = next((p.resolve() for p in candidate_paths if p.exists()), None)
if notebook_file is None:
    raise FileNotFoundError(f"Notebook not found: {notebook_name}")

dest_dir = notebook_file.parent
exec_pdf = dest_dir / "analise_northwind_executivo.pdf"
assets_dir = dest_dir / "executivo_assets"
assets_dir.mkdir(exist_ok=True)

# Intermediate HTML — used only to generate the PDF, deleted afterwards.
_tmp_html = dest_dir / "_exec_tmp.html"


def fmt_cur_short(value):
    if abs(value) >= 1_000_000:
        return f'R$ {value/1_000_000:.1f}M'
    if abs(value) >= 1_000:
        return f'R$ {value/1_000:.1f}K'
    return f'R$ {value:,.0f}'


# Fallback: rebuild the model from CSVs if the kernel was restarted.
if "fact_sales" not in globals():
    print("Kernel has no variables. Loading local CSV data...")
    data_dir = dest_dir / "data"
    orders = pd.read_csv(data_dir / "orders.csv", sep=";")
    order_details = pd.read_csv(data_dir / "order_details.csv", sep=";")
    products = pd.read_csv(data_dir / "products.csv", sep=";")
    categories = pd.read_csv(data_dir / "categories.csv", sep=";")
    employees = pd.read_csv(data_dir / "employees.csv", sep=";")

    for col in ["order_date", "required_date", "shipped_date"]:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

    fact_sales = order_details.merge(
        orders[["order_id", "customer_id", "employee_id", "order_date", "required_date", "shipped_date"]],
        on="order_id",
        how="left",
    )
    fact_sales["gross_sales"] = fact_sales["unit_price"] * fact_sales["quantity"]
    fact_sales["net_sales"] = fact_sales["gross_sales"] * (1 - fact_sales["discount"])
    fact_sales["discount_value"] = fact_sales["gross_sales"] - fact_sales["net_sales"]
    fact_sales["shipping_delay_days"] = (fact_sales["shipped_date"] - fact_sales["required_date"]).dt.days
    fact_sales["shipping_delay_days"] = fact_sales["shipping_delay_days"].fillna(0)
    fact_sales["shipped_on_time_flag"] = (fact_sales["shipping_delay_days"] <= 0).astype(int)

    # category_name inline in dim_product (matches Gold schema)
    dim_product = products.merge(
        categories[["category_id", "category_name"]], on="category_id", how="left"
    )
    dim_employee = employees[["employee_id", "first_name", "last_name"]].copy()

# Base KPIs
receita_liquida = float(fact_sales["net_sales"].sum())
total_pedidos = int(fact_sales["order_id"].nunique())
total_clientes = int(fact_sales["customer_id"].nunique())
ticket_medio = receita_liquida / total_pedidos if total_pedidos else 0.0
pedidos_prazo = int((fact_sales["shipped_on_time_flag"] == 1).sum())
total_pedidos_entrega = int(len(fact_sales))
pct_prazo = (pedidos_prazo / total_pedidos_entrega * 100.0) if total_pedidos_entrega else 0.0
atraso_medio = float(fact_sales["shipping_delay_days"].mean()) if total_pedidos_entrega else 0.0

# Plotly PNG export dependency.
try:
    import kaleido  # noqa: F401
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "kaleido", "-q"], check=False)


def save_plot(fig, filename):
    out = assets_dir / filename
    fig.write_image(str(out), format="png", width=1400, height=800, scale=2)
    return out.name


# ── Chart data ────────────────────────────────────────────────────────────────
receita_mes = fact_sales.groupby(fact_sales["order_date"].dt.to_period("M")).agg({
    "net_sales": "sum",
    "order_id": "nunique",
}).reset_index()
receita_mes["data"] = receita_mes["order_date"].dt.to_timestamp()
receita_mes = receita_mes.drop("order_date", axis=1).rename(columns={
    "net_sales": "receita",
    "order_id": "pedidos",
})

# category_name already inline in dim_product (Gold schema)
fact_cat = fact_sales.merge(dim_product[["product_id", "category_name"]], on="product_id", how="left")
receita_cat = fact_cat.groupby("category_name")["net_sales"].sum().sort_values(ascending=False)

pedidos_desc = fact_sales.groupby("order_id").agg({"net_sales": "sum", "discount": "mean"}).reset_index()

fact_prod = fact_sales.merge(dim_product[["product_id", "product_name"]], on="product_id", how="left")
top_prod = fact_prod.groupby("product_name")["net_sales"].sum().sort_values(ascending=False).head(15)

data_ref = fact_sales["order_date"].max()
ultimo_pedido = fact_sales.groupby("customer_id")["order_date"].max()
dias_inativo = (data_ref - ultimo_pedido).dt.days
dias_df = pd.DataFrame({"dias_inativo": dias_inativo})

fact_emp = fact_sales.merge(dim_employee[["employee_id", "first_name", "last_name"]], on="employee_id", how="left")
fact_emp["salesperson"] = fact_emp["first_name"].fillna("") + " " + fact_emp["last_name"].fillna("")
receita_vendedor = fact_emp.groupby("salesperson")["net_sales"].sum().sort_values(ascending=False)

# ── Derived metrics ───────────────────────────────────────────────────────────
top_categoria = receita_cat.index[0]
top_categoria_valor = float(receita_cat.iloc[0])
top_categoria_pct = (top_categoria_valor / receita_liquida * 100.0) if receita_liquida else 0.0
media_desconto = float(pedidos_desc["discount"].mean())
media_ticket_pedido = float(pedidos_desc["net_sales"].mean())
top_produto = top_prod.index[0]
top_produto_valor = float(top_prod.iloc[0])
top_vendedor = receita_vendedor.index[0]
top_vendedor_valor = float(receita_vendedor.iloc[0])
clientes_em_risco_pct = (int((dias_inativo > 90).sum()) / total_clientes * 100.0) if total_clientes else 0.0
atraso_mediano = float(dias_df["dias_inativo"].median())
atraso_medio_neg = float(fact_sales.loc[fact_sales["shipping_delay_days"] < 0, "shipping_delay_days"].mean()) if (fact_sales["shipping_delay_days"] < 0).any() else 0.0
atraso_medio_pos = float(fact_sales.loc[fact_sales["shipping_delay_days"] > 0, "shipping_delay_days"].mean()) if (fact_sales["shipping_delay_days"] > 0).any() else 0.0

# ── Charts ────────────────────────────────────────────────────────────────────
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=receita_mes["data"],
    y=receita_mes["receita"],
    mode="lines+markers",
    line=dict(color="#0B6E4F", width=3),
    marker=dict(size=7),
    name="Revenue",
))
fig1.update_layout(title="Monthly Net Revenue", xaxis_title="Period", yaxis_title="Revenue (R$)")
if not receita_mes.empty:
    fig1.add_annotation(
        x=receita_mes["data"].iloc[-1],
        y=receita_mes["receita"].iloc[-1],
        text=f"Last month: {fmt_cur_short(receita_mes['receita'].iloc[-1])}",
        showarrow=True, arrowhead=2, ax=40, ay=-30, bgcolor="white",
    )

fig2 = px.bar(
    x=receita_cat.values, y=receita_cat.index, orientation="h",
    title="Revenue by Category", labels={"x": "Revenue (R$)", "y": "Category"},
    color=receita_cat.values, color_continuous_scale="Tealgrn",
    text=[fmt_cur_short(v) for v in receita_cat.values],
)
fig2.update_traces(textposition="inside", insidetextanchor="middle", cliponaxis=False)
fig2.update_yaxes(autorange="reversed")

fig3 = px.scatter(
    pedidos_desc, x="discount", y="net_sales",
    title="Discount vs. Order Value",
    labels={"discount": "Discount Rate", "net_sales": "Value (R$)"},
    opacity=0.65, color="discount", color_continuous_scale="Reds",
)

fig4 = px.bar(
    x=top_prod.values, y=top_prod.index, orientation="h",
    title="Top 15 Products by Revenue", labels={"x": "Revenue (R$)", "y": "Product"},
    color=top_prod.values, color_continuous_scale="Blues",
    text=[fmt_cur_short(v) for v in top_prod.values],
)
fig4.update_traces(textposition="inside", insidetextanchor="middle", cliponaxis=False)
fig4.update_yaxes(autorange="reversed")

fig5 = px.histogram(
    dias_df, x="dias_inativo", nbins=40,
    title="Days Without a Purchase — Distribution",
    labels={"dias_inativo": "Days Inactive", "count": "Customers"},
    color_discrete_sequence=["#0B6E4F"],
)
fig5.add_vline(x=90, line_dash="dash", line_color="red")

fig6 = px.bar(
    x=receita_vendedor.head(10).values, y=receita_vendedor.head(10).index, orientation="h",
    title="Top 10 Salespersons by Revenue", labels={"x": "Revenue (R$)", "y": "Salesperson"},
    color=receita_vendedor.head(10).values, color_continuous_scale="Plasma",
    text=[fmt_cur_short(v) for v in receita_vendedor.head(10).values],
)
fig6.update_traces(textposition="inside", insidetextanchor="middle", cliponaxis=False)
fig6.update_yaxes(autorange="reversed")

fig7 = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=float(pct_prazo),
    title={"text": "% On-Time Deliveries"},
    delta={"reference": 95},
    gauge={
        "axis": {"range": [0, 100]},
        "bar": {"color": "#0B6E4F"},
        "steps": [
            {"range": [0, 50], "color": "#ECECEC"},
            {"range": [50, 90], "color": "#DCEFE8"},
        ],
    },
))

fig8 = px.histogram(
    fact_sales, x="shipping_delay_days", nbins=40,
    title="Delays and Early Deliveries — Distribution",
    labels={"shipping_delay_days": "Delay (days)", "count": "Orders"},
    color_discrete_sequence=["#1F77B4"],
)
fig8.add_vline(x=0, line_dash="dash", line_color="green")

# ── Export PNGs and build HTML ────────────────────────────────────────────────
print("Generating static charts (PNG)...")
charts_data = [
    (save_plot(fig1, "01_monthly_revenue.png"),    "Monthly Net Revenue",               f"Last period reading: {fmt_cur_short(receita_mes['receita'].iloc[-1])}."),
    (save_plot(fig2, "02_revenue_by_category.png"),"Revenue by Category",               f"{top_categoria} leads with {fmt_cur_short(top_categoria_valor)} ({top_categoria_pct:.1f}% of total)."),
    (save_plot(fig3, "03_discount_vs_order.png"),  "Discount vs. Order Value",          f"Avg discount: {media_desconto:.1%}. Avg order value: {fmt_cur_short(media_ticket_pedido)}."),
    (save_plot(fig4, "04_top_products.png"),        "Top 15 Products by Revenue",        f"Top product: {top_produto} — {fmt_cur_short(top_produto_valor)}."),
    (save_plot(fig5, "05_customer_inactivity.png"), "Customer Inactivity — Distribution",f"Median inactivity: {atraso_mediano:.0f} days. {clientes_em_risco_pct:.1f}% of base at risk (>90d)."),
    (save_plot(fig6, "06_top_salespersons.png"),    "Top 10 Salespersons by Revenue",    f"{top_vendedor} leads with {fmt_cur_short(top_vendedor_valor)}."),
    (save_plot(fig7, "07_on_time_gauge.png"),       "% On-Time Deliveries",              f"Current level: {pct_prazo:.2f}% (target: 95%)."),
    (save_plot(fig8, "08_delay_distribution.png"),  "Delays and Early Deliveries",       f"Avg early: {atraso_medio_neg:.1f}d. Avg late: {atraso_medio_pos:.1f}d."),
]

kpi_cards = [
    ("Net Revenue",          f"R$ {receita_liquida:,.2f}"),
    ("Total Orders",         f"{total_pedidos:,}"),
    ("Avg Order Value",      f"R$ {ticket_medio:,.2f}"),
    ("Active Customers",     f"{total_clientes:,}"),
    ("% On-Time Deliveries", f"{pct_prazo:.2f}%"),
    ("Avg Delay",            f"{atraso_medio:.2f} days"),
]

cards_html = "\n".join([
    f"<div class='card'><h3>{escape(k)}</h3><p>{escape(v)}</p></div>"
    for k, v in kpi_cards
])

charts_html = "\n".join([
    f"<section class='chart'>"
    f"<h2>{escape(title)}</h2>"
    f"<p class='chart-text'>{escape(desc)}</p>"
    f"<img src='executivo_assets/{escape(img)}' alt='{escape(title)}'>"
    f"</section>"
    for img, title, desc in charts_data
])

html_content = f"""<!doctype html>
<html lang='en'>
<head>
  <meta charset='utf-8'>
  <title>Northwind Traders - Executive Presentation</title>
  <style>
    @page {{ size: A4 portrait; margin: 12mm; }}
    body {{ font-family: Segoe UI, Calibri, Arial, sans-serif; margin: 0; color: #1B1F24; background: #F7FAF9; }}
    .wrap {{ max-width: 1080px; margin: 0 auto; padding: 16px 18px 24px; }}
    h1 {{ margin: 0 0 8px; font-size: 30px; color: #0B6E4F; }}
    .sub {{ margin: 0 0 20px; color: #4B5563; font-size: 16px; line-height: 1.4; }}
    .cards {{ display: grid; grid-template-columns: repeat(3, 1fr); gap: 12px; margin-bottom: 18px; }}
    .card {{ background: #fff; border: 1px solid #D7E5DE; border-radius: 12px; padding: 12px; }}
    .card h3 {{ margin: 0 0 6px; font-size: 13px; color: #4B5563; font-weight: 600; }}
    .card p {{ margin: 0; font-size: 23px; color: #0F172A; font-weight: 700; }}
    .chart {{ background: #fff; border: 1px solid #D7E5DE; border-radius: 12px; padding: 12px; margin: 0 0 14px; page-break-inside: avoid; }}
    .chart h2 {{ margin: 0 0 6px; font-size: 18px; color: #0F172A; }}
    .chart-text {{ margin: 0 0 10px; font-size: 14px; color: #4B5563; line-height: 1.45; }}
    .chart img {{ width: 100%; height: auto; display: block; }}
    .footer {{ margin-top: 8px; font-size: 12px; color: #6B7280; }}
  </style>
</head>
<body>
  <div class='wrap'>
    <h1>Northwind Traders — Executive Presentation</h1>
    <p class='sub'>Consolidated dashboard for executive decision-making: commercial performance, customers and logistics.</p>
    <div class='cards'>{cards_html}</div>
    {charts_html}
    <p class='footer'>Report generated automatically from the analytical notebook.</p>
  </div>
</body>
</html>"""

_tmp_html.write_text(html_content, encoding="utf-8")

# ── Generate PDF via Playwright ───────────────────────────────────────────────
print("Generating executive PDF...")
pdf_script = f'''
from pathlib import Path
from playwright.sync_api import sync_playwright

html_file = Path(r"{_tmp_html}").resolve()
pdf_file  = Path(r"{exec_pdf}").resolve()

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page(viewport={{"width": 1400, "height": 2200}})
    page.goto(html_file.as_uri(), wait_until="networkidle")
    page.wait_for_selector("img", timeout=30000)
    page.pdf(
        path=str(pdf_file),
        format="A4",
        print_background=True,
        margin={{"top": "10mm", "right": "10mm", "bottom": "10mm", "left": "10mm"}}
    )
    browser.close()

print("OK: Executive PDF generated at " + str(pdf_file))
'''

run_pdf = subprocess.run([sys.executable, "-c", pdf_script], capture_output=True, text=True)

# Remove the intermediate HTML regardless of the outcome.
_tmp_html.unlink(missing_ok=True)

if run_pdf.returncode == 0 and exec_pdf.exists():
    print(run_pdf.stdout.strip())
    print(f"Charts exported: {len(charts_data)}")
else:
    print("Failed to generate executive PDF.")
    print(run_pdf.stderr[:2000])

Gerando gráficos estáticos (PNG)...
Gerando PDF executivo...
OK: PDF executivo gerado em C:\Users\rafae\Documents\Carreira\challenge_indicium\analise_northwind_executivo.pdf
Gráficos exportados: 8
